# From Quantum Math to Molecular Sovereignty
Interactive notebook for exploring saved VQE PES results without re-running full simulations.

## 1. Introduction to Quantum Chemistry
Ground-state electronic energies define reaction thermodynamics and molecular stability.

## 2. Second Quantization
$$\hat{H} = \sum_{pq} h_{pq} a^\dagger_p a_q + \frac{1}{2}\sum_{pqrs} g_{pqrs} a^\dagger_p a^\dagger_q a_r a_s$$

## 3. Mapping and Variational Workflow
Parity mapping converts fermionic operators to qubits, then VQE minimizes energy over ansatz parameters.

In [ ]:
from pathlib import Path
import json
import sys

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

results_dir = project_root / 'results' / 'raw_data'

def discover_results():
    files = sorted(results_dir.glob('pes_*.json'))
    mapping = {}
    for file in files:
        name = file.stem.replace('pes_', '')
        if name.endswith('_table'):
            continue
        mapping[name] = file
    return mapping

def load_result(molecule):
    files = discover_results()
    path = files.get(molecule)
    if path is None:
        return None
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

available = discover_results()
print('Available saved results:', ', '.join(available.keys()) if available else 'none')

## 4. Interactive Results Explorer
Use widgets to switch molecule, ansatz, plot type, and convergence bond length.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

CHEM_ACC = 0.0016

molecule_options = sorted(list(available.keys())) if available else ['H2']
molecule_dd = widgets.Dropdown(options=molecule_options, value=molecule_options[0], description='Molecule')
ansatz_dd = widgets.Dropdown(options=['UCCSD', 'EfficientSU2'], value='UCCSD', description='Ansatz')
plot_dd = widgets.Dropdown(options=['PES', 'Error', 'Convergence'], value='PES', description='Plot')
bond_slider = widgets.SelectionSlider(options=[1.4], value=1.4, description='Bond (A)', continuous_update=False)
show_acc = widgets.Checkbox(value=True, description='Show chemical accuracy')
log_err = widgets.Checkbox(value=False, description='Log-scale error')
refresh_btn = widgets.Button(description='Refresh files', button_style='info')

def safe_pairwise(distances, values):
    pairs = []
    for d, v in zip(distances, values):
        if v is None:
            continue
        try:
            f = float(v)
        except Exception:
            continue
        if math.isnan(f):
            continue
        pairs.append((float(d), f))
    return pairs

def update_controls(*_):
    data = load_result(molecule_dd.value)
    if not data:
        ansatz_dd.options = ['UCCSD', 'EfficientSU2']
        bond_slider.options = [1.4]
        bond_slider.value = 1.4
        return
    ansatz_names = list((data.get('vqe_energies') or {}).keys())
    if not ansatz_names:
        ansatz_names = ['UCCSD', 'EfficientSU2']
    ansatz_dd.options = ansatz_names
    if ansatz_dd.value not in ansatz_names:
        ansatz_dd.value = ansatz_names[0]

    bonds = [round(float(b), 3) for b in data.get('distances', [])]
    if not bonds:
        bonds = [1.4]
    bond_slider.options = bonds
    if bond_slider.value not in bonds:
        bond_slider.value = min(bonds, key=lambda x: abs(x - 1.4))

def refresh_files(_):
    files = discover_results()
    keys = sorted(files.keys())
    if keys:
        molecule_dd.options = keys
        if molecule_dd.value not in keys:
            molecule_dd.value = keys[0]
    update_controls()

refresh_btn.on_click(refresh_files)
molecule_dd.observe(update_controls, names='value')

def render_plot(molecule, ansatz, plot_kind, bond_value, show_threshold, log_error):
    data = load_result(molecule)
    if not data:
        print(f'No saved result file for {molecule}. Run: python -m src.pes_generator --molecule {molecule}')
        return

    d = [float(x) for x in data.get('distances', [])]
    exact = [float(x) for x in data.get('exact_energies', [])]
    vqe = [float(x) for x in (data.get('vqe_energies', {}).get(ansatz, []))]

    plt.figure(figsize=(10, 6))

    if plot_kind == 'PES':
        exact_pairs = safe_pairwise(d, exact)
        vqe_pairs = safe_pairwise(d, vqe)
        if exact_pairs:
            plt.plot([x for x, _ in exact_pairs], [y for _, y in exact_pairs], label='Exact', linewidth=2)
        if vqe_pairs:
            plt.plot([x for x, _ in vqe_pairs], [y for _, y in vqe_pairs], label=ansatz, linestyle='--', marker='o')
        plt.ylabel('Energy (Hartree)')
        plt.title(f'{molecule}: PES (Exact vs {ansatz})')

    elif plot_kind == 'Error':
        errors = []
        xvals = []
        for idx, bond in enumerate(d):
            if idx < len(exact) and idx < len(vqe):
                if not (math.isnan(exact[idx]) or math.isnan(vqe[idx])):
                    xvals.append(bond)
                    errors.append(abs(vqe[idx] - exact[idx]))
        if xvals:
            plt.plot(xvals, errors, marker='o', linewidth=1.8, label=f'{ansatz} error')
        if show_threshold:
            plt.axhline(CHEM_ACC, color='red', linestyle='--', label='Chemical accuracy')
        if log_error:
            plt.yscale('log')
        plt.ylabel('Absolute Error (Hartree)')
        plt.title(f'{molecule}: Error Curve ({ansatz})')

    else:  # Convergence
        key = f'{float(bond_value):.3f}'
        history = (data.get('histories', {}).get(ansatz, {}) or {}).get(key, [])
        clean = [h for h in history if isinstance(h, dict) and 'iteration' in h and 'energy' in h]
        if not clean:
            plt.text(0.1, 0.5, f'No convergence trace for {ansatz} at {key} A', fontsize=12)
            plt.axis('off')
        else:
            it = [int(h['iteration']) for h in clean]
            en = [float(h['energy']) for h in clean]
            plt.plot(it, en, linewidth=1.8, marker='.')
            plt.ylabel('Energy (Hartree)')
            plt.title(f'{molecule}: Convergence {ansatz} at {key} A')
            idx = d.index(float(key)) if float(key) in d else None
            if idx is not None and idx < len(exact) and not math.isnan(exact[idx]):
                plt.axhline(exact[idx], color='black', linestyle='--', label='Exact')
                plt.legend()
        plt.xlabel('Iteration')

    if plot_kind != 'Convergence':
        plt.xlabel('Bond Length (Angstrom)')
        plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

update_controls()
controls = widgets.VBox([
    widgets.HBox([molecule_dd, ansatz_dd, plot_dd]),
    widgets.HBox([bond_slider, show_acc, log_err, refresh_btn]),
])
out = widgets.interactive_output(
    render_plot,
    {
        'molecule': molecule_dd,
        'ansatz': ansatz_dd,
        'plot_kind': plot_dd,
        'bond_value': bond_slider,
        'show_threshold': show_acc,
        'log_error': log_err,
    },
)
display(controls, out)

## 5. Optional Re-run Cell
Use this only if a result file is missing and you want to regenerate it.

In [ ]:
# Optional regenerate step
# from src.pes_generator import PESGenerator
# import yaml
# with open(project_root / 'config' / 'simulation_config.yaml', 'r', encoding='utf-8') as f:
#     config = yaml.safe_load(f)
# PESGenerator(config).run('H2')
# PESGenerator(config).run('LiH')